# Task-1

In [34]:
import pandas as pd
import sqlite3

In [35]:
conn = sqlite3.connect("sales.db")

In [36]:
# Load the three CSV files into SQLite tables
for name in ["sales", "customers", "products"]:
    pd.read_csv(f"{name}.csv").to_sql(
        name,
        conn,
        index=False,
        if_exists="replace"
    )

In [37]:
# Helper function to run SQL queries
def run(sql):
    return pd.read_sql_query(sql, conn)

In [38]:
#Q1: Show the first 10 rows of the sales table.
run("SELECT * FROM sales LIMIT 10")

,order_id,date,customer_id,product_id,quantity
0,O00001,31-10-2024,C999,P019,2
1,O00002,14-03-2024,C999,P014,1
2,O00003,21-10-2024,C999,P004,1
3,O00004,03-01-2024,C999,P003,3
4,O00005,18-10-2024,C999,P018,4
5,O00006,15-10-2024,C010,P006,2
6,O00007,12-10-2024,C013,P016,5
7,O00008,31-08-2024,C046,P003,2
8,O00009,21-06-2024,C048,P006,5
9,O00010,15-09-2024,C003,P006,2


In [39]:
#Count the total number of orders.
run("SELECT COUNT(*) AS total_orders FROM sales")

,total_orders
0,1000


In [40]:
#Show all orders with a quantity of 5.
run("SELECT * FROM sales WHERE quantity = 5")

,order_id,date,customer_id,product_id,quantity
0,O00007,12-10-2024,C013,P016,5
1,O00009,21-06-2024,C048,P006,5
2,O00014,16-06-2024,C015,P019,5
3,O00020,11-02-2024,C025,P015,5
4,O00022,01-09-2024,C005,P020,5
...,...,...,...,...,...
186,O00981,02-12-2024,C042,P009,5
187,O00988,28-12-2024,C034,P004,5
188,O00989,26-08-2024,C016,P005,5
189,O00990,21-02-2024,C001,P010,5


In [41]:
# List all products in the 'Electronics' category.
run("SELECT * FROM products WHERE category = 'Electronics'")

,product_id,product_name,category,unit_price
0,P001,Laptop,Electronics,800
1,P002,Mouse,Electronics,25
2,P003,Keyboard,Electronics,45
3,P004,Monitor,Electronics,250
4,P005,Webcam,Electronics,60
5,P006,Headphones,Electronics,120
6,P007,USB Hub,Electronics,35
7,P016,Cable Set,Electronics,30
8,P017,Power Bank,Electronics,50
9,P018,Speaker,Electronics,90


In [42]:
#Show the 5 most expensive products (by unit price).
run("SELECT * FROM products ORDER BY unit_price DESC LIMIT 5")

,product_id,product_name,category,unit_price
0,P001,Laptop,Electronics,800
1,P010,Standing Desk,Office,400
2,P020,Tablet,Electronics,350
3,P004,Monitor,Electronics,250
4,P009,Chair,Office,180


In [43]:
# Count how many customers are in each region.
run("SELECT region, COUNT(*) AS customer_count FROM customers GROUP BY region")

,region,customer_count
0,East,13
1,North,12
2,South,11
3,West,16


In [44]:
run("SELECT product_id, SUM(quantity) AS total_quantity FROM sales GROUP BY product_id")

,product_id,total_quantity
0,P001,126
1,P002,138
2,P003,159
3,P004,177
4,P005,171
5,P006,145
6,P007,159
7,P008,178
8,P009,156
9,P010,172


In [45]:
#Find the average unit price per category.
run("SELECT category, AVG(unit_price) AS avg_unit FROM products GROUP BY category")

,category,avg_unit
0,Accessories,31.000000
1,Electronics,163.750000
2,Office,206.666667
3,Stationery,11.500000


In [46]:
# Count orders per product, but only show products with more than 50 orders (use HAVING).
run("SELECT product_id, COUNT(*) AS order_count FROM sales GROUP BY product_id HAVING COUNT(*) > 50")

,product_id,order_count
0,P003,56
1,P004,62
2,P005,54
3,P007,53
4,P008,59
5,P010,54
6,P017,51
7,P019,57
8,P020,53


In [47]:
#Find the single highest and lowest unit price in the products table.
run("SELECT MAX(unit_price) AS highest_price , MIN(unit_price) AS lowest_price FROM products")

,highest_price,lowest_price
0,800,8


In [48]:
# Join sales to products and show each order's product name and price.
run("SELECT s.order_id, s.product_id, p.product_name, p.unit_price FROM sales AS s JOIN products AS p ON s.product_id=p.product_id")

,order_id,product_id,product_name,unit_price
0,O00001,P019,Microphone,110
1,O00002,P014,Water Bottle,20
2,O00003,P004,Monitor,250
3,O00004,P003,Keyboard,45
4,O00005,P018,Speaker,90
...,...,...,...,...
995,O00996,P016,Cable Set,30
996,O00997,P004,Monitor,250
997,O00998,P007,USB Hub,35
998,O00999,P004,Monitor,250


In [49]:
#Compute total revenue (quantity times price) across all sales.
run("SELECT SUM(s.quantity * p.unit_price) AS total_revenue FROM sales AS s JOIN products AS p ON s.product_id = p.product_id")

,total_revenue
0,405150


In [50]:
#Compute total revenue per region (join all three tables).
run("SELECT c.region , SUM(s.quantity * p.unit_price) AS total_revenue FROM sales AS s JOIN products AS p ON s.product_id=p.product_id JOIN customers AS c ON s.customer_id = c.customer_id GROUP BY c.region")

,region,total_revenue
0,East,121636
1,North,83444
2,South,81814
3,West,117271


In [51]:
# Compute total revenue per product category.
run("SELECT p.category , SUM(s.quantity * p.unit_price) AS total_revenue FROM sales AS s JOIN products AS p ON s.product_id=p.product_id GROUP BY category")

,category,total_revenue
0,Accessories,12081
1,Electronics,285690
2,Office,104000
3,Stationery,3379


In [52]:
#Find how many sales reference a customer who isn't in the customers table (hint: LEFT JOIN + IS NULL).
run("SELECT COUNT(*) AS missing_customer FROM sales AS s LEFT JOIN customers AS c ON s.customer_id = c.customer_id WHERE c.customer_id IS NULL")

,missing_customer
0,5


In [53]:
#Rank all products by total revenue using a window function.
run("""
WITH product_revenue AS (
    SELECT
        p.product_id,
        p.product_name,
        SUM(s.quantity * p.unit_price) AS total_revenue
    FROM sales AS s
    JOIN products AS p
    ON s.product_id = p.product_id
    GROUP BY p.product_id, p.product_name
)
SELECT
    product_id,
    product_name,
    total_revenue,
    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS revenue_rank
FROM product_revenue
ORDER BY revenue_rank
""")


,product_id,product_name,total_revenue,revenue_rank
0,P001,Laptop,100800,1
1,P010,Standing Desk,68800,2
2,P020,Tablet,54600,3
3,P004,Monitor,44250,4
4,P009,Chair,28080,5
5,P019,Microphone,18590,6
6,P006,Headphones,17400,7
7,P018,Speaker,11520,8
8,P005,Webcam,10260,9
9,P017,Power Bank,8050,10


In [54]:
#Compute a running total of revenue by month.
run("""
WITH monthly_revenue AS (
    SELECT
        substr(sales.date, 7, 4) || '-' ||
        substr(sales.date, 4, 2) AS month,

        SUM(sales.quantity * products.unit_price) AS revenue

    FROM sales
    JOIN products
        ON sales.product_id = products.product_id

    GROUP BY
        substr(sales.date, 7, 4) || '-' ||
        substr(sales.date, 4, 2)
)

SELECT
    month,
    revenue,
    SUM(revenue) OVER (
        ORDER BY month
    ) AS running_total

FROM monthly_revenue

ORDER BY month
""")

,month,revenue,running_total
0,2024-01,32561,32561
1,2024-02,47995,80556
2,2024-03,38608,119164
3,2024-04,28702,147866
4,2024-05,27291,175157
5,2024-06,31720,206877
6,2024-07,34522,241399
7,2024-08,31247,272646
8,2024-09,29860,302506
9,2024-10,30620,333126


In [55]:
#Find the top-selling product within each category.
run("""WITH product_sales AS (
    SELECT
        products.category,
        products.product_id,
        products.product_name,
        SUM(sales.quantity) AS total_quantity
    FROM sales
    JOIN products
        ON sales.product_id = products.product_id
    GROUP BY
        products.category,
        products.product_id,
        products.product_name
),
ranked_products AS (
    SELECT
        category,
        product_id,
        product_name,
        total_quantity,
        RANK() OVER (
            PARTITION BY category
            ORDER BY total_quantity DESC
        ) AS product_rank
    FROM product_sales
)
SELECT
    category,
    product_id,
    product_name,
    total_quantity
FROM ranked_products
WHERE product_rank = 1""")

,category,product_id,product_name,total_quantity
0,Accessories,P015,Phone Stand,137
1,Electronics,P004,Monitor,177
2,Office,P008,Desk Lamp,178
3,Stationery,P011,Notebook,158


In [56]:
#Calculate each region's revenue and its percentage of total revenue.
run("""WITH region_revenue AS (
    SELECT
        customers.region,
        SUM(sales.quantity * products.unit_price) AS revenue
    FROM sales
    JOIN customers
        ON sales.customer_id = customers.customer_id
    JOIN products
        ON sales.product_id = products.product_id
    GROUP BY customers.region
)
SELECT
    region,
    revenue,
    ROUND(
        revenue * 100.0 / SUM(revenue) OVER (),
        2
    ) AS revenue_percentage
FROM region_revenue;""")

,region,revenue,revenue_percentage
0,East,121636,30.10
1,North,83444,20.65
2,South,81814,20.24
3,West,117271,29.02


# Task-2

In [60]:
# Load CSV files
sales = pd.read_csv("sales.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

In [61]:
# Merge sales with customers to get region
sales_customers = sales.merge(
    customers,
    on="customer_id",
    how="left"
)

In [62]:
# Merge with products to get unit price
full = sales_customers.merge(
    products,
    on="product_id",
    how="left"
)

In [63]:
# Calculate revenue
full["revenue"] = full["quantity"] * full["unit_price"]

In [64]:
# Calculate total revenue for each region
pandas_result = (
    full.groupby("region")["revenue"]
    .sum()
    .reset_index()
)

In [65]:
# Sort result
pandas_result = pandas_result.sort_values(
    "region"
).reset_index(drop=True)

print("Pandas Result:")
display(pandas_result)


Pandas Result:


,region,revenue
0,East,121636
1,North,83444
2,South,81814
3,West,117271


In [66]:
run("""
SELECT
    customers.region,
    SUM(sales.quantity * products.unit_price) AS revenue
FROM sales
JOIN customers
    ON sales.customer_id = customers.customer_id
JOIN products
    ON sales.product_id = products.product_id
GROUP BY customers.region
ORDER BY customers.region;
""")

,region,revenue
0,East,121636
1,North,83444
2,South,81814
3,West,117271


In [ ]:
#Pandas was easier to write because merge() and groupby() made the analysis simple and step-by-step.
#SQL was easier to read because the JOIN and GROUP BY clearly showed how the tables were connected.
#I would choose Pandas for data cleaning and analysis in Python, and SQL when working with data stored in a database, especially for large datasets.

# Task-3

In [69]:
#Compute a running total of revenue over the whole year, ordered by date (a cumulative sum).
run("""
SELECT
    sales.date,
    sales.order_id,
    sales.quantity * products.unit_price AS revenue,

    SUM(sales.quantity * products.unit_price)
        OVER (
            ORDER BY sales.date, sales.order_id
        ) AS running_total

FROM sales
JOIN products
    ON sales.product_id = products.product_id

ORDER BY sales.date, sales.order_id;
""")

,date,order_id,revenue,running_total
0,01-02-2024,O00653,75,75
1,01-02-2024,O00781,4000,4075
2,01-03-2024,O00087,140,4215
3,01-03-2024,O00250,32,4247
4,01-03-2024,O00367,105,4352
...,...,...,...,...
995,31-10-2024,O00001,220,399610
996,31-10-2024,O00288,1400,401010
997,31-10-2024,O00445,50,401060
998,31-12-2024,O00301,4000,405060


In [70]:
# Rank customers by their total spending, showing the top 10.
run("""
WITH customer_spending AS (
    SELECT
        customers.customer_id,
        customers.customer_name,
        SUM(sales.quantity * products.unit_price) AS total_spending
    FROM sales
    JOIN customers
        ON sales.customer_id = customers.customer_id
    JOIN products
        ON sales.product_id = products.product_id
    GROUP BY
        customers.customer_id,
        customers.customer_name
)

SELECT
    customer_id,
    customer_name,
    total_spending,
    RANK() OVER (
        ORDER BY total_spending DESC
    ) AS spending_rank
FROM customer_spending
ORDER BY spending_rank
LIMIT 10;
""")

,customer_id,customer_name,total_spending,spending_rank
0,C024,Customer_24,19300,1
1,C015,Customer_15,17480,2
2,C003,Customer_3,12885,3
3,C039,Customer_39,12568,4
4,C023,Customer_23,12464,5
5,C043,Customer_43,12214,6
6,C027,Customer_27,11775,7
7,C050,Customer_50,11644,8
8,C007,Customer_7,11114,9
9,C029,Customer_29,10887,10


In [71]:
# For each region, rank its customers by spending using PARTITION BY, so ranking restarts per region.
run("""
WITH customer_spending AS (
    SELECT
        customers.region,
        customers.customer_id,
        customers.customer_name,
        SUM(sales.quantity * products.unit_price) AS total_spending
    FROM sales
    JOIN customers
        ON sales.customer_id = customers.customer_id
    JOIN products
        ON sales.product_id = products.product_id
    GROUP BY
        customers.region,
        customers.customer_id,
        customers.customer_name
)

SELECT
    region,
    customer_id,
    customer_name,
    total_spending,

    RANK() OVER (
        PARTITION BY region
        ORDER BY total_spending DESC
    ) AS regional_rank

FROM customer_spending

ORDER BY region, regional_rank;
""")

,region,customer_id,customer_name,total_spending,regional_rank
0,East,C015,Customer_15,17480,1
1,East,C003,Customer_3,12885,2
2,East,C043,Customer_43,12214,3
3,East,C029,Customer_29,10887,4
4,East,C034,Customer_34,10843,5
5,East,C049,Customer_49,9920,6
6,East,C021,Customer_21,9745,7
7,East,C008,Customer_8,8986,8
8,East,C017,Customer_17,5967,9
9,East,C011,Customer_11,5905,10


In [72]:
#Compute, for each month, that month's revenue AND the previous month's revenue side by side (look up theLAG window function).
run("""
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', sales.date) AS month,
        SUM(sales.quantity * products.unit_price) AS revenue
    FROM sales
    JOIN products
        ON sales.product_id = products.product_id
    GROUP BY strftime('%Y-%m', sales.date)
)

SELECT
    month,
    revenue,

    LAG(revenue) OVER (
        ORDER BY month
    ) AS previous_month_revenue

FROM monthly_revenue
ORDER BY month;
""")

,month,revenue,previous_month_revenue
0,None,405150,None


In [73]:
#Using LAG, compute month-over-month growth: the percentage change in revenue from one month to the next
run("""
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', sales.date) AS month,
        SUM(sales.quantity * products.unit_price) AS revenue
    FROM sales
    JOIN products
        ON sales.product_id = products.product_id
    GROUP BY strftime('%Y-%m', sales.date)
),

monthly_with_previous AS (
    SELECT
        month,
        revenue,

        LAG(revenue) OVER (
            ORDER BY month
        ) AS previous_month_revenue

    FROM monthly_revenue
)

SELECT
    month,
    revenue,
    previous_month_revenue,

    ROUND(
        (revenue - previous_month_revenue)
        * 100.0
        / previous_month_revenue,
        2
    ) AS mom_growth_percentage

FROM monthly_with_previous
ORDER BY month;
""")

,month,revenue,previous_month_revenue,mom_growth_percentage
0,None,405150,None,None
